In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pretty_midi

In [3]:
#Check GPU enabled
import torch
import pandas   as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil
print(torch.cuda.is_available())    

True


In [4]:
from pathlib import Path
import shutil

# midi_directory and excluded_directory are assumed to be Path objects or strings
midi_directory = Path("/content/drive/MyDrive/capstone_team_3/MidiDatasets/590-Classical-music-midi")
test_external_directory = Path("/content/drive/MyDrive/capstone_team_3/MidiDatasets/TestingSamples/MidiOutputs")

In [8]:
# Load the MIDI files and extract features using pretty_midi
import pretty_midi
import os   
midi_files = []
for root, dirs, files in os.walk(midi_directory):
    for file in files:
        if file.endswith('.mid') or file.endswith('.midi'):
            midi_files.append(os.path.join(root, file))
print(f"Total MIDI files found: {len(midi_files)}")

Total MIDI files found: 292


In [25]:
# Load the external test MIDI files and extract features using pretty_midi
test_midi_files = []
for root, dirs, files in os.walk(test_external_directory):
    for file in files:
        if file.endswith('.mid') or file.endswith('.midi'):
            test_midi_files.append(os.path.join(root, file))
print(f"Total external test MIDI files found: {len(test_midi_files)}")

Total external test MIDI files found: 50


In [6]:
#  extract features from MIDI data with start,end,pitch,velocity
import numpy as np
def extract_features(midi_data):
        features = []
        previous_pitch = None
        for instrument in midi_data.instruments:
            for note in instrument.notes:
                #  duration, pitch, velocity, pitch_diff    
                pitch_diff = note.pitch - previous_pitch if previous_pitch is not None else 0
                features.append([note.start,note.end,note.pitch,note.velocity])
                previous_pitch = note.pitch
        return np.array(features)   

In [10]:
midi_features = []
for midi_path in midi_files:
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    features = extract_features(midi_data)
    midi_features.append(features)
print(f"Extracted features from {len(midi_features)} MIDI files.")

Extracted features from 292 MIDI files.


In [26]:
# Extract features from external test MIDI files
test_midi_features = []
for midi_path in test_midi_files:
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    features = extract_features(midi_data)
    test_midi_features.append(features)
print(f"Extracted features from {len(test_midi_features)} external test MIDI files.")

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Extracted features from 50 external test MIDI files.


In [11]:
# Convert the list of features into a format suitable for training (e.g., numpy array)
midi_features_array = np.array(midi_features, dtype=object)  # Using dtype=object to handle variable-length feature arrays
print(f"Shape of MIDI features array: {midi_features_array.shape}")
#Store the extracted features for later use as CSV or binary file
import pandas as pd
# Convert the list of features into a DataFrame for easier storage and manipulation
features_df = pd.DataFrame(midi_features_array, columns=['features'])
# Add file names or identifiers if needed (optional)
features_df['file_name'] = [os.path.basename(path) for path in midi_files]  
# Save the DataFrame to picklet format for efficient storage
features_df.to_pickle('/content/drive/MyDrive/capstone_team_3/midi_features_for_transformet.pkl')
print("MIDI features saved to pickle file.") 

Shape of MIDI features array: (292,)
MIDI features saved to pickle file.


In [27]:
#  Convert the list of features into a format suitable for training (e.g., numpy array)
test_midi_features_array = np.array(test_midi_features, dtype=object)  #Using dtype=object to handle variable-length feature arrays
print(f"Shape of external test MIDI features array: {test_midi_features_array.shape}")

features_test_df = pd.DataFrame(test_midi_features_array, columns=['features'])
features_test_df['file_name'] = [os.path.basename(path) for path in test_midi_files]
features_test_df.to_pickle('/content/drive/MyDrive/capstone_team_3/test_midi_features_for_transformer.pkl')
print("External test MIDI features saved to pickle file.")  

Shape of external test MIDI features array: (50,)
External test MIDI features saved to pickle file.


In [7]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=np.float32)
    vec2 = np.asarray(vec2, dtype=np.float32)
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return float(dot_product / (norm_vec1 * norm_vec2))

def features_to_vector(features):
    features = np.asarray(features, dtype=np.float32)
    if features.ndim == 1:
        features = features.reshape(1, -1)
    summary_vector = np.concatenate([
        features.mean(axis=0),
        features.std(axis=0),
        features.min(axis=0),
        features.max(axis=0)
    ])
    return summary_vector.astype(np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [14]:
#load the features from the pickle file
loaded_features_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/midi_features_for_transformet.pkl')
print("MIDI features loaded from pickle file.")
print(f"Shape of loaded MIDI features DataFrame: {loaded_features_df.shape}")

MIDI features loaded from pickle file.
Shape of loaded MIDI features DataFrame: (292, 2)


In [29]:
# load the test features from the pickle file
loaded_test_features_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/test_midi_features_for_transformer.pkl')
print("External test MIDI features loaded from pickle file.")
print(f"Shape of loaded external test MIDI features DataFrame: {loaded_test_features_df.shape}")

External test MIDI features loaded from pickle file.
Shape of loaded external test MIDI features DataFrame: (50, 2)


In [8]:
# Convert the pitch into  windowed sequences for training a sequence model
def create_sequences(features, sequence_length=256):
    sequences = []
    for i in range(len(features) - sequence_length):
        seq = features[i:i + sequence_length]
        sequences.append(seq)
    return np.array(sequences)

In [ ]:

# use loaded_features_df to create sequences for training
# Covvert the all the midi features into sequences for training,  Each sequence will be a window of 50 notes (or other features) for training a model like an LSTM or Transformer. Label the sequences appropriately for supervised learning, give the name of the midi file as the label for each sequence.
all_sequences = []
for midi_path, features in zip(loaded_features_df['file_name'], loaded_features_df['features']):
    if features.size > 0:  # Check if there are features to avoid errors
        sequences = create_sequences(features)
        file_name = os.path.basename(midi_path)  # Get the file name from the path
        labels = [file_name] * len(sequences)  # Label each sequence with the MIDI file name
        all_sequences.extend(zip(sequences, labels))
print(f"Total sequences created: {len(all_sequences)}")

# Create a DF for the sequences and labels
sequences_df = pd.DataFrame(all_sequences, columns=['sequence', 'label'])
print(f"Shape of sequences DataFrame: {sequences_df.shape}")

Total sequences created: 630987
Shape of sequences DataFrame: (630987, 2)


In [30]:
# create sequences for the test features as well
test_sequences = []
for midi_path, features in zip(loaded_test_features_df['file_name'], loaded_test_features_df['features']):
    if features.size > 0:  # Check if there are features to avoid errors
        sequences = create_sequences(features)
        file_name = os.path.basename(midi_path)  # Get the file name from the path
        labels = [file_name] * len(sequences)  # Label each sequence with the MIDI file name
        test_sequences.extend(zip(sequences, labels))
print(f"Total test sequences created: {len(test_sequences)}")   

# Create a DF for the test sequences and labels
test_sequences_df = pd.DataFrame(test_sequences, columns=['sequence', 'label'])
print(f"Shape of test sequences DataFrame: {test_sequences_df.shape}")  

Total test sequences created: 203723
Shape of test sequences DataFrame: (203723, 2)


In [16]:
# Store the sequences and labels for training
sequences_df.to_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256_trans_features.pkl')
print("MIDI sequences saved to pickle file.")  

MIDI sequences saved to pickle file.


In [31]:
# Store the test sequences and labels for testing
test_sequences_df.to_pickle('/content/drive/MyDrive/capstone_team_3/data_set/test_classic_midi_sequences_256_trans_features.pkl')
print("External test MIDI sequences saved to pickle file.")

External test MIDI sequences saved to pickle file.


### Pretrained Music-BERT style model + fine-tuning
This section builds note-token inputs from the same sequence features (`start`, `end`, `pitch`, `velocity`) and fine-tunes a pretrained BERT classifier for label prediction.

In [13]:
# Load the sequences and labels for training
loaded_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256_trans_features.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {loaded_sequences_df.shape}")  

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (630987, 2)


In [9]:
# Install / import transformers stack
try:
    import transformers
except ImportError:
    import sys
    !{sys.executable} -m pip install -q transformers accelerate

from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
print(f"transformers version: {transformers.__version__}")


transformers version: 5.0.0


In [5]:
 # Save label_encoder
label_encoder_save_path = '/content/drive/MyDrive/capstone_team_3/saved_models/transformer_label_encoder_v1'
import pickle
with open(label_encoder_save_path, 'wb') as f:
    pickle.dump(label_encoder, f)   
    

NameError: name 'label_encoder' is not defined

In [10]:
# Tokenisation: map each note's [start, end, pitch, velocity] into a single
# discrete token ID within BERT's vocabulary (size 30522).
#   start_bin   : 0-9   (0.5 s buckets, capped at 5 s)
#   end_bin     : 0-9   (0.5 s buckets, capped at 5 s)
#   pitch_bin   : 0-127 (direct MIDI pitch)
#   velocity_bin: 0-9   (0-127 -> 10 bins)
# token_id = 1000 + start_bin*1280 + end_bin*128 + pitch_bin*10 + velocity_bin
# Max token_id ≈ 14951 — well within BERT vocab size of 30522.

BERT_VOCAB_SIZE = 30522
TOKEN_OFFSET = 1000

def sequenceToTokens(seq_features, max_length=128):
    """Convert a (seq_len, 4) array of [start, end, pitch, velocity]
    into token IDs and an attention mask."""
    tokens = []
    for note in seq_features:
        start, end, pitch, velocity = note
        start_bin    = min(int(abs(start) / 0.5), 9)
        end_bin      = min(int(abs(end)   / 0.5), 9)
        pitch_bin    = min(int(pitch), 127)
        velocity_bin = min(int(abs(velocity) // 13), 9)
        token_id = TOKEN_OFFSET + start_bin * 1280 + end_bin * 128 + pitch_bin * 10 + velocity_bin
        tokens.append(min(token_id, BERT_VOCAB_SIZE - 1))

    tokens = tokens[:max_length]
    attention_mask = [1] * len(tokens)
    if len(tokens) < max_length:
        pad_len = max_length - len(tokens)
        tokens        += [0] * pad_len
        attention_mask += [0] * pad_len
    return tokens, attention_mask

print("sequenceToTokens defined.")


sequenceToTokens defined.


In [14]:
# Train / test split and label encoding
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(loaded_sequences_df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)}  Test: {len(test_df)}")

label_encoder = LabelEncoder()
label_encoder.fit(loaded_sequences_df['label'])
y_train = label_encoder.transform(train_df['label'])
y_test  = label_encoder.transform(test_df['label'])
num_classes     = len(label_encoder.classes_)
sequence_length = loaded_sequences_df['sequence'].iloc[0].shape[0]
print(f"Classes: {num_classes}  Sequence length: {sequence_length}")


Train: 504789  Test: 126198
Classes: 289  Sequence length: 256


In [15]:
# Load the sequences and labels for  testing
test_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/test_classic_midi_sequences_256_trans_features.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {test_sequences_df.shape}")  

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (203723, 2)


In [ ]:
# label encode the test labels
# y_test_external = label_encoder.transform(test_sequences_df['label'])

In [22]:
# Tokenise sequences and build DataLoaders
max_len = min(128, sequence_length)
batch_size = 16

In [ ]:


train_ids, train_masks = [], []
for seq in train_df['sequence']:
    ids, mask = sequenceToTokens(seq, max_length=max_len)
    train_ids.append(ids)
    train_masks.append(mask)

test_ids, test_masks = [], []
for seq in test_df['sequence']:
    ids, mask = sequenceToTokens(seq, max_length=max_len)
    test_ids.append(ids)
    test_masks.append(mask)

train_input_ids       = torch.tensor(train_ids,   dtype=torch.long)
train_attention_masks = torch.tensor(train_masks, dtype=torch.long)
train_labels          = torch.tensor(y_train,     dtype=torch.long)

test_input_ids        = torch.tensor(test_ids,    dtype=torch.long)
test_attention_masks  =  torch.tensor(test_masks,  dtype=torch.long)
test_labels           = torch.tensor(y_test,      dtype=torch.long)

assert train_input_ids.size(0) == train_labels.size(0), "Train size mismatch"
assert test_input_ids.size(0)  == test_labels.size(0),  "Test size mismatch"

print(f"Train: ids={train_input_ids.shape}, labels={train_labels.shape}")
print(f"Test:  ids={test_input_ids.shape},  labels={test_labels.shape}")
print(f"Token ID range: {train_input_ids[train_input_ids > 0].min().item()} – {train_input_ids.max().item()}")


train_loader_bert = DataLoader(
    TensorDataset(train_input_ids, train_attention_masks, train_labels),
    batch_size=batch_size, shuffle=True
)
test_loader_bert = DataLoader(
    TensorDataset(test_input_ids, test_attention_masks, test_labels),
    batch_size=batch_size, shuffle=False
)
print(f"Train batches: {len(train_loader_bert)}  Test batches: {len(test_loader_bert)}")


Train: ids=torch.Size([504789, 128]), labels=torch.Size([504789])
Test:  ids=torch.Size([126198, 128]),  labels=torch.Size([126198])
Token ID range: 1244 – 14748
Train batches: 31550  Test batches: 7888


In [23]:
#  External test set preparation: Tokenise the external test sequences and prepare a DataLoader for inference
test_external_ids, test_external_masks = [], []
for seq in test_sequences_df['sequence']:
    ids, mask = sequenceToTokens(seq, max_length=max_len)
    test_external_ids.append(ids)
    test_external_masks.append(mask)    
test_external_input_ids       = torch.tensor(test_external_ids,   dtype=torch.long)
test_external_attention_masks = torch.tensor(test_external_masks, dtype=torch.long)
print(f"External test: ids={test_external_input_ids.shape}, attention_masks={test_external_attention_masks.shape}") 

test_external_loader = DataLoader(
    TensorDataset(test_external_input_ids, test_external_attention_masks),
    batch_size=batch_size, shuffle=False
)
print(f"External test batches: {len(test_external_loader)}")

External test: ids=torch.Size([203723, 128]), attention_masks=torch.Size([203723, 128])
External test batches: 12733


In [16]:
# Load pretrained model (music-BERT → BERT fallback)
bert_candidates = ['microsoft/musicbert-base', 'wazenmai/MIDI-BERT', 'google-bert/bert-base-uncased']
bert_model_name, bert_model, last_err = None, None, None

for candidate in bert_candidates:
    try:
        cfg = AutoConfig.from_pretrained(candidate, num_labels=num_classes)
        bert_model = AutoModelForSequenceClassification.from_pretrained(
            candidate, config=cfg, ignore_mismatched_sizes=True
        )
        bert_model_name = candidate
        break
    except Exception as e:
        last_err = e

if bert_model is None:
    raise RuntimeError(f"Could not load any pretrained model: {last_err}")

bert_model = bert_model.to(device)
print(f"Loaded checkpoint: {bert_model_name}")
print(f"Model parameters: {sum(p.numel() for p in bert_model.parameters()):,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded checkpoint: google-bert/bert-base-uncased
Model parameters: 109,704,481


In [ ]:
# Fine-tuning loop
bert_epochs = 3
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)
total_steps = len(train_loader_bert) * bert_epochs
bert_scheduler = get_linear_schedule_with_warmup(
    bert_optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps
)

bert_history = []
for epoch in range(bert_epochs):
    bert_model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for input_ids, attention_mask, labels in train_loader_bert:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        bert_optimizer.zero_grad()
        out = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        bert_optimizer.step()
        bert_scheduler.step()

        train_loss    += out.loss.item() * labels.size(0)
        train_correct += (out.logits.argmax(dim=1) == labels).sum().item()
        train_total   += labels.size(0)

    bert_model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in test_loader_bert:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)
            out = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            test_loss    += out.loss.item() * labels.size(0)
            test_correct += (out.logits.argmax(dim=1) == labels).sum().item()
            test_total   += labels.size(0)

    metrics = {
        'epoch':          epoch + 1,
        'train_loss':     train_loss    / max(1, train_total),
        'train_accuracy': train_correct / max(1, train_total),
        'test_loss':      test_loss     / max(1, test_total),
        'test_accuracy':  test_correct  / max(1, test_total),
        'checkpoint':     bert_model_name,
    }
    bert_history.append(metrics)
    print(
        f"Epoch {epoch+1}/{bert_epochs} - "
        f"train_loss: {metrics['train_loss']:.4f}, train_acc: {metrics['train_accuracy']:.4f}, "
        f"test_loss: {metrics['test_loss']:.4f}, test_acc: {metrics['test_accuracy']:.4f}"
    )

bert_history_df = pd.DataFrame(bert_history)
bert_history_df


In [ ]:
# Save the modle and tokenizer for later use
model_save_path = '/content/drive/MyDrive/capstone_team_3/saved_models/transformer_model_v1'
tokenizer_save_path = '/content/drive/MyDrive/capstone_team_3/saved_models/transformer_tokenizer_v1'

# Save it in google drive for later use
# Note: The actual saving code will depend on the model and tokenizer you are using. Below is a placeholder for saving a Hugging Face transformer model and tokenizer.

bert_model.save_pretrained(model_save_path)

In [17]:
# Load the model, tokenizer, and label encoder for inference
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pickle

# label_encoder_save_path = '/content/drive/MyDrive/capstone_team_3/saved_models/transformer_label_encoder_v1'
# with open(label_encoder_save_path, 'rb') as f:
#     label_encoder = pickle.load(f)  

model_save_path = '/content/drive/MyDrive/capstone_team_3/saved_models/transformer_model_v1'
bert_model = AutoModelForSequenceClassification.from_pretrained(model_save_path)    



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [26]:
# Verify the model on the external test set
# bert_model.eval()
# model_device = next(bert_model.parameters()).device
# print(f"Running external inference on: {model_device}")

# external_test_preds = []
# with torch.no_grad():
#     for input_ids, attention_mask in test_external_loader:
#         input_ids = input_ids.to(model_device)
#         attention_mask = attention_mask.to(model_device)
#         out = bert_model(input_ids=input_ids, attention_mask=attention_mask)
#         external_test_preds.extend(out.logits.argmax(dim=1).cpu().numpy())

# external_test_predicted_labels = label_encoder.inverse_transform(np.array(external_test_preds, dtype=int))
# test_sequences_df['predicted_label'] = external_test_predicted_labels
# print("External predictions created.")
# test_sequences_df[['label', 'predicted_label']].head()


In [25]:
# Verify the model on the external test set (GPU inference)
# Force GPU - re-check CUDA availability at inference time
import tqdm
if torch.cuda.is_available():
    inference_device = torch.device('cuda')
    torch.cuda.empty_cache()
else:
    inference_device = torch.device('cpu')
    print("WARNING: CUDA not available, falling back to CPU")

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Running external inference on: {inference_device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  Memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

bert_model = bert_model.to(inference_device)
bert_model.eval()

external_test_preds = []
with torch.no_grad():
    for input_ids, attention_mask in tqdm.tqdm(test_external_loader):
        input_ids      = input_ids.to(inference_device)
        attention_mask = attention_mask.to(inference_device)
        out = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        external_test_preds.extend(out.logits.argmax(dim=1).cpu().numpy())

external_test_predicted_labels = label_encoder.inverse_transform(np.array(external_test_preds, dtype=int))
test_sequences_df['predicted_label'] = external_test_predicted_labels
print(f"External predictions complete: {len(external_test_preds)} sequences")
test_sequences_df[['label', 'predicted_label']].head()


CUDA available: True
Running external inference on: cuda
GPU: NVIDIA A100-SXM4-40GB  Memory free: 37.7 GB


100%|██████████| 12733/12733 [05:44<00:00, 36.99it/s]

External predictions complete: 203723 sequences


,label,predicted_label
0,Sonata op35 n4 .mid,islamei.mid
1,Sonata op35 n4 .mid,islamei.mid
2,Sonata op35 n4 .mid,islamei.mid
3,Sonata op35 n4 .mid,islamei.mid
4,Sonata op35 n4 .mid,islamei.mid


In [28]:
# Save external test predictions to Google Drive
import json

results_path = '/content/drive/MyDrive/capstone_team_3/results/external_test_predictions_transformer_v1.pkl'
test_sequences_df[['label', 'predicted_label']].to_pickle(results_path)
print(f"External test predictions saved to {results_path}")

# Also save model metadata (works even if training history is not in current kernel)
has_history = 'bert_history_df' in globals() and isinstance(bert_history_df, pd.DataFrame) and len(bert_history_df) > 0

if has_history:
    final_epoch = bert_history_df.iloc[-1]
    results_block = {
        "final_train_loss":     round(float(final_epoch['train_loss']), 4),
        "final_train_accuracy": round(float(final_epoch['train_accuracy']), 4),
        "final_test_loss":      round(float(final_epoch['test_loss']), 4),
        "final_test_accuracy":  round(float(final_epoch['test_accuracy']), 4),
    }
    training_history_block = bert_history_df.to_dict(orient='records')
else:
    results_block = {
        "note": "Training history unavailable in current session. Run the fine-tuning cell first to include train/test metrics."
    }
    training_history_block = []

transformer_meta = {
    "modelName": "Transformer (BERT fine-tuned)",
    "tokenization": "start+end+pitch+velocity",
    "checkpoint": globals().get('bert_model_name', None),
    "hyperParams": {
        "epochs": globals().get('bert_epochs', None),
        "batch_size": globals().get('batch_size', None),
        "learning_rate": 2e-5,
        "max_token_length": globals().get('max_len', None),
        "optimizer": "AdamW",
        "scheduler": "linear_warmup",
    },
    "dataParams": {
        "sequence_length": globals().get('sequence_length', None),
        "num_classes": globals().get('num_classes', None),
        "train_size": len(train_df) if 'train_df' in globals() else None,
        "test_size": len(test_df) if 'test_df' in globals() else None,
        "external_test_size": len(test_sequences_df),
    },
    "results": results_block,
    "trainingHistory": training_history_block,
}

meta_path = '/content/drive/MyDrive/capstone_team_3/models/transformer_v1_meta.json'
with open(meta_path, 'w') as f:
    json.dump(transformer_meta, f, indent=2)

print(f"Model metadata saved to {meta_path}")
print(json.dumps({k: v for k, v in transformer_meta.items() if k != 'trainingHistory'}, indent=2))


External test predictions saved to /content/drive/MyDrive/capstone_team_3/results/external_test_predictions_transformer_v1.pkl
Model metadata saved to /content/drive/MyDrive/capstone_team_3/models/transformer_v1_meta.json
{
  "modelName": "Transformer (BERT fine-tuned)",
  "tokenization": "start+end+pitch+velocity",
  "checkpoint": "google-bert/bert-base-uncased",
  "hyperParams": {
    "epochs": null,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "max_token_length": 128,
    "optimizer": "AdamW",
    "scheduler": "linear_warmup"
  },
  "dataParams": {
    "sequence_length": 256,
    "num_classes": 289,
    "train_size": 504789,
    "test_size": 126198,
    "external_test_size": 203723
  },
  "results": {
    "note": "Training history unavailable in current session. Run the fine-tuning cell first to include train/test metrics."
  }
}


## Alternative Model: Pitch + Duration + Velocity Tokens
This model uses a simpler 3-feature tokenization: `[pitch, duration, velocity]` instead of `[start, end, pitch, velocity]`.


In [29]:
# Define tokenization for [pitch, duration, velocity]
# pitch_bin   : 0-127 (direct MIDI pitch)
# duration_bin: 0-9   (duration in 0.5s buckets, capped at 5s)
# velocity_bin: 0-9   (velocity 0-127 -> 10 bins)
# token_id = 2000 + pitch_bin*100 + duration_bin*10 + velocity_bin
# Max token_id ≈ 2000 + 127*100 + 9*10 + 9 = 14699 — safe within BERT vocab.

def sequenceToTokens_v2(seq_features, max_length=128):
    """
    Convert a (seq_len, 4) array of [start, end, pitch, velocity]
    into tokens using [pitch, duration, velocity].
    duration = end - start
    """
    tokens = []
    for note in seq_features:
        start, end, pitch, velocity = note
        duration = max(end - start, 0.01)  # Avoid zero duration
        
        pitch_bin    = min(int(pitch), 127)
        duration_bin = min(int(duration / 0.5), 9)
        velocity_bin = min(int(abs(velocity) // 13), 9)
        
        token_id = 2000 + pitch_bin * 100 + duration_bin * 10 + velocity_bin
        tokens.append(min(token_id, BERT_VOCAB_SIZE - 1))

    tokens = tokens[:max_length]
    attention_mask = [1] * len(tokens)
    if len(tokens) < max_length:
        pad_len = max_length - len(tokens)
        tokens        += [0] * pad_len
        attention_mask += [0] * pad_len
    return tokens, attention_mask

print("sequenceToTokens_v2 (pitch+duration+velocity) defined.")


sequenceToTokens_v2 (pitch+duration+velocity) defined.


In [30]:
# Tokenise sequences using v2 (pitch+duration+velocity) and build DataLoaders
train_ids_v2, train_masks_v2 = [], []
for seq in train_df['sequence']:
    ids, mask = sequenceToTokens_v2(seq, max_length=max_len)
    train_ids_v2.append(ids)
    train_masks_v2.append(mask)

test_ids_v2, test_masks_v2 = [], []
for seq in test_df['sequence']:
    ids, mask = sequenceToTokens_v2(seq, max_length=max_len)
    test_ids_v2.append(ids)
    test_masks_v2.append(mask)

train_input_ids_v2       = torch.tensor(train_ids_v2,   dtype=torch.long)
train_attention_masks_v2 = torch.tensor(train_masks_v2, dtype=torch.long)

test_input_ids_v2        = torch.tensor(test_ids_v2,    dtype=torch.long)
test_attention_masks_v2  = torch.tensor(test_masks_v2,  dtype=torch.long)

assert train_input_ids_v2.size(0) == train_labels.size(0), "Train size mismatch (v2)"
assert test_input_ids_v2.size(0)  == test_labels.size(0),  "Test size mismatch (v2)"

print(f"v2 Train: ids={train_input_ids_v2.shape}, labels={train_labels.shape}")
print(f"v2 Test:  ids={test_input_ids_v2.shape},  labels={test_labels.shape}")
print(f"v2 Token ID range: {train_input_ids_v2[train_input_ids_v2 > 0].min().item()} – {train_input_ids_v2.max().item()}")

train_loader_bert_v2 = DataLoader(
    TensorDataset(train_input_ids_v2, train_attention_masks_v2, train_labels),
    batch_size=batch_size, shuffle=True
)
test_loader_bert_v2 = DataLoader(
    TensorDataset(test_input_ids_v2, test_attention_masks_v2, test_labels),
    batch_size=batch_size, shuffle=False
)


print(f"v2 Train batches: {len(train_loader_bert_v2)}  Test batches: {len(test_loader_bert_v2)}")


NameError: name 'train_labels' is not defined

In [ ]:
# external test loader external seq
ext_test_ids_v2, ext_test_masks_v2 = [], []
for seq in test_sequences_df['sesequencequ']
    ids, mask = sequenceToTokens_v2(seq, max_length=max_len)
    ext_test_ids_v2.append(ids)
    ext_test_masks_v2.append(mask)

ext_test_input_ids_v2        = torch.tensor(ext_test_ids_v2,    dtype=torch.long)
ext_test_attention_masks_v2  = torch.tensor(ext_test_masks_v2,  dtype=torch.long)

test_loader_bert_v2 = DataLoader(
    TensorDataset(ext_test_input_ids_v2, ext_test_attention_masks_v2, test_labels),
    batch_size=batch_size, shuffle=False
)

In [ ]:
# Load pretrained model for v2
bert_model_v2, bert_model_name_v2, last_err_v2 = None, None, None

for candidate in bert_candidates:
    try:
        cfg = AutoConfig.from_pretrained(candidate, num_labels=num_classes)
        bert_model_v2 = AutoModelForSequenceClassification.from_pretrained(
            candidate, config=cfg, ignore_mismatched_sizes=True
        )
        bert_model_name_v2 = candidate
        break
    except Exception as e:
        last_err_v2 = e

if bert_model_v2 is None:
    raise RuntimeError(f"Could not load pretrained model for v2: {last_err_v2}")

bert_model_v2 = bert_model_v2.to(device)
print(f"v2 Loaded checkpoint: {bert_model_name_v2}")
print(f"v2 Model parameters: {sum(p.numel() for p in bert_model_v2.parameters()):,}")


In [ ]:
# Fine-tuning loop for v2
bert_optimizer_v2 = torch.optim.AdamW(bert_model_v2.parameters(), lr=2e-5)
total_steps_v2 = len(train_loader_bert_v2) * bert_epochs
bert_scheduler_v2 = get_linear_schedule_with_warmup(
    bert_optimizer_v2,
    num_warmup_steps=max(1, total_steps_v2 // 10),
    num_training_steps=total_steps_v2
)

bert_history_v2 = []
for epoch in range(bert_epochs):
    bert_model_v2.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for input_ids, attention_mask, labels in train_loader_bert_v2:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        bert_optimizer_v2.zero_grad()
        out = bert_model_v2(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        bert_optimizer_v2.step()
        bert_scheduler_v2.step()

        train_loss    += out.loss.item() * labels.size(0)
        train_correct += (out.logits.argmax(dim=1) == labels).sum().item()
        train_total   += labels.size(0)

    bert_model_v2.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in test_loader_bert_v2:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)
            out = bert_model_v2(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            test_loss    += out.loss.item() * labels.size(0)
            test_correct += (out.logits.argmax(dim=1) == labels).sum().item()
            test_total   += labels.size(0)

    metrics_v2 = {
        'epoch':          epoch + 1,
        'train_loss':     train_loss    / max(1, train_total),
        'train_accuracy': train_correct / max(1, train_total),
        'test_loss':      test_loss     / max(1, test_total),
        'test_accuracy':  test_correct  / max(1, test_total),
        'checkpoint':     bert_model_name_v2,
    }
    bert_history_v2.append(metrics_v2)
    print(
        f"v2 Epoch {epoch+1}/{bert_epochs} - "
        f"train_loss: {metrics_v2['train_loss']:.4f}, train_acc: {metrics_v2['train_accuracy']:.4f}, "
        f"test_loss: {metrics_v2['test_loss']:.4f}, test_acc: {metrics_v2['test_accuracy']:.4f}"
    )

bert_history_df_v2 = pd.DataFrame(bert_history_v2)
bert_history_df_v2


In [ ]:
# Model comparison: v1 vs v2
comparison_df = pd.DataFrame({
    'Model': ['v1 (start+end+pitch+velocity)', 'v2 (pitch+duration+velocity)'],
    'Final Train Loss': [bert_history_df.iloc[-1]['train_loss'], bert_history_df_v2.iloc[-1]['train_loss']],
    'Final Train Acc': [bert_history_df.iloc[-1]['train_accuracy'], bert_history_df_v2.iloc[-1]['train_accuracy']],
    'Final Test Loss': [bert_history_df.iloc[-1]['test_loss'], bert_history_df_v2.iloc[-1]['test_loss']],
    'Final Test Acc': [bert_history_df.iloc[-1]['test_accuracy'], bert_history_df_v2.iloc[-1]['test_accuracy']],
    'Checkpoint': [bert_model_name, bert_model_name_v2],
})
print("Model Comparison:")
print(comparison_df.to_string(index=False))
